# V17 – Kryptografie (Teil 1): Theorie

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- die Grundbegriffe **Klartext**, **Geheimtext**, **Schlüssel**, **Verschlüsseln** und **Entschlüsseln** präzise verwenden,
- das **Kerckhoffs-Prinzip** erklären und auf moderne Verfahren übertragen,
- die **Caesar-Chiffre** manuell und per Formel `(pos + k) % 26` anwenden,
- einen **Brute-Force-Angriff** auf Caesar gedanklich und in Python durchspielen,
- **symmetrische** Verfahren (z. B. AES) von **asymmetrischen** Verfahren (z. B. RSA) unterscheiden und das **Hybrid-Verfahren** bei HTTPS einordnen,
- den Unterschied zwischen **Block-Chiffre** und **Stream-Chiffre** auf Schlagwortniveau benennen.

## ⏱️ Zeitbudget
Ca. 30 Minuten konzentriertes Lesen und Ausprobieren.

## 🧭 TL;DR
Kryptografie sorgt dafür, dass Daten zwischen Alice und Bob vertraulich bleiben. Die **Caesar-Chiffre** ist das einfachste Verfahren, per Brute-Force aber in Sekunden gebrochen. Moderne Verfahren unterscheiden **symmetrische** (beide nutzen denselben Schlüssel, z. B. AES) und **asymmetrische** Verfahren (öffentlicher und privater Schlüssel, z. B. RSA). HTTPS kombiniert beides zu einem **Hybrid-Verfahren**.

## 📦 Voraussetzungen
- [00_python_recap.ipynb](00_python_recap.ipynb) (Strings, `%`, `ord`/`chr`)
- Grundkenntnisse Funktionen/Schleifen (V06, V10)

## 1. Warum Verschlüsselung? Drei ingenieurnahe Szenarien

Drei Szenarien aus dem Maschinenbau-Alltag machen klar, warum Kryptografie keine reine Theorie-Disziplin ist, sondern eine Kernkompetenz moderner Ingenieurinnen und Ingenieure.

**Szenario 1 – Industriespionage.** Ein Konstruktionsbüro entwickelt einen neuen Getriebetyp. Die CAD-Zeichnungen werden per E-Mail zwischen Standorten ausgetauscht. Ohne Verschlüsselung liegen sie auf jedem Mailserver im Klartext vor – ein einziger korrupter Administrator genügt, und ein Wettbewerber ist zwei Jahre Entwicklungsvorsprung billig eingekauft.

**Szenario 2 – DSGVO und Personendaten.** Eine Maschine protokolliert, welcher Mitarbeiter wann welche Schicht gefahren hat. Diese Log-Dateien enthalten **personenbezogene Daten** im Sinne der **DSGVO** und müssen bei Übertragung und Lagerung verschlüsselt sein. Verstoß = Bußgeld bis 4 % des Jahresumsatzes.

**Szenario 3 – Maschinenkommandos.** Eine CNC-Fräse empfängt Befehle über das Werksnetz. Wenn die Kommandos unverschlüsselt und ungesichert laufen, kann ein Angreifer Vorschübe ändern und damit **Ausschuss erzeugen** oder sogar Menschen gefährden. Die Verschlüsselung schützt hier nicht nur die Vertraulichkeit, sondern über digitale Signaturen auch die **Integrität** der Kommandos.

### Drei Schutzziele auf einen Blick

Die klassische **Kryptografie** verfolgt traditionell drei zentrale Schutzziele, die in der Literatur oft als **CIA-Triade** (Confidentiality, Integrity, Authenticity) abgekürzt werden:

1. **Vertraulichkeit** – Nur Berechtigte können den Inhalt lesen.
2. **Integrität** – Eine Veränderung der Nachricht unterwegs fällt auf.
3. **Authentizität** – Der Empfänger ist sicher, wer der Absender war.

In V17 konzentrieren wir uns auf das erste Ziel, die **Vertraulichkeit**. Die beiden anderen Ziele tauchen in V18 (Hashes, Signaturen) wieder auf.

## 2. Die Grundbegriffe

Um sauber über Kryptografie sprechen zu können, brauchst du fünf Begriffe, die in jeder Vorlesung und jedem Fachbuch vorkommen:

- **Klartext** (englisch *plaintext*) – Die lesbare Originalnachricht, z. B. `"HALLO WELT"`.
- **Geheimtext** (englisch *ciphertext*) – Die unleserliche, verschlüsselte Version, z. B. `"KDOOR ZHOW"`.
- **Schlüssel** (englisch *key*) – Ein Parameter, der aus Klartext Geheimtext macht (und umgekehrt). Bei Caesar ist der Schlüssel einfach eine Zahl zwischen 1 und 25.
- **Verschlüsseln** – Der Vorgang *Klartext + Schlüssel* → *Geheimtext*.
- **Entschlüsseln** – Der Vorgang *Geheimtext + Schlüssel* → *Klartext*.

Die beiden Personen, zwischen denen die Nachricht läuft, heißen in der Fachliteratur traditionell **Alice** (Senderin) und **Bob** (Empfänger). Die Angreiferin heißt **Eve** (von *eavesdropper*, Lauscherin) oder **Mallory** (aktiver Angreifer).

In [1]:
from IPython.display import Markdown, display
with open("diagramme/01_klartext_geheimtext.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))

```mermaid
flowchart LR
    K(["Klartext<br/>z. B. HALLO"]) --> V["Verschluesseln<br/>mit Schluessel"]
    V --> G(["Geheimtext<br/>z. B. KDOOR"])
    G --> E["Entschluesseln<br/>mit Schluessel"]
    E --> K2(["Klartext<br/>HALLO"])

```

### Der Schlüssel – das eigentliche Geheimnis

Wichtig: **Das Geheime ist nicht der Algorithmus, sondern der Schlüssel.** Bei Caesar wissen alle, wie das Verfahren funktioniert (Buchstaben verschieben). Trotzdem kann Eve die Nachricht nicht lesen, wenn sie den **Schlüsselwert** (also die Anzahl der Verschiebeschritte) nicht kennt. Die Sicherheit hängt ausschließlich daran, dass der Schlüssel geheim bleibt.

Diese Trennung zwischen öffentlichem Verfahren und geheimem Schlüssel ist so fundamental, dass sie einen eigenen Namen hat.

## 3. Das Kerckhoffs-Prinzip (1883)

Der niederländische Kryptograf **Auguste Kerckhoffs** formulierte bereits 1883 ein Prinzip, das heute in jedem sicheren System gilt:

> *Ein kryptografisches Verfahren muss auch dann sicher bleiben, wenn alles außer dem Schlüssel öffentlich bekannt ist.*

Das klingt erst paradox – warum sollte man einem Angreifer verraten, wie das Verfahren arbeitet? Die Begründung ist pragmatisch: **Ein Algorithmus lässt sich nicht dauerhaft geheim halten.** Er ist in Software implementiert, läuft auf hunderten Rechnern und wird früher oder später reverse-engineered. Der **Schlüssel** hingegen ist nur eine kurze Zahlenfolge, die sich leicht schützen, austauschen und bei Verdacht neu setzen lässt.

Der Gegensatz dazu heißt *Security through Obscurity* – Sicherheit durch Verschleierung. Wer darauf setzt, hat früher oder später ein Problem, denn aus Obscurity wird irgendwann eine veröffentlichte CVE-Meldung.

### Konsequenzen für moderne Systeme

Aus Kerckhoffs' Prinzip folgen drei praktische Regeln, an die sich jedes seriöse Security-Team hält:

1. **Nutze nur standardisierte, öffentlich geprüfte Algorithmen** wie AES, RSA, SHA-256. Eigenbau-Krypto ist fast immer angreifbar.
2. **Halte die Schlüssel geheim**, nicht den Code. Quelltext darf offen sein – Schlüssel gehören in ein **Key-Management-System** oder ein Hardware-Sicherheitsmodul (HSM).
3. **Rotiere Schlüssel regelmäßig.** Je länger ein Schlüssel in Gebrauch ist, desto größer die Angriffsfläche.

## 4. Die Caesar-Chiffre

Die **Caesar-Chiffre** ist das älteste dokumentierte Verschlüsselungsverfahren und geht auf den römischen Feldherrn **Gaius Julius Caesar** zurück, der damit militärische Befehle gegen neugierige Boten schützte. Das Prinzip ist verblüffend einfach: Jeder Buchstabe wird im Alphabet um eine feste Anzahl Stellen nach rechts verschoben.

Bei einer Verschiebung um drei Stellen wird aus `A` ein `D`, aus `B` ein `E`, aus `C` ein `F` und so weiter. Am Ende des Alphabets **rollt die Zählung rund**: Aus `X` wird `A`, aus `Y` wird `B`, aus `Z` wird `C`. Genau dieses „Rundrollen“ ist der Grund, warum wir später den Modulo-Operator brauchen.

In [2]:
from IPython.display import Markdown, display
with open("diagramme/02_caesar_verschiebung.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))

```mermaid
flowchart TD
    S(["Caesar-Verschiebung um 3"]) --> A
    A["A"] --> AD["D"]
    B["B"] --> BE["E"]
    C["C"] --> CF["F"]
    DD["..."] --> DDD["..."]
    W["W"] --> WZ["Z"]
    X["X"] --> XA["A"]
    Y["Y"] --> YB["B"]
    Z["Z"] --> ZC["C"]

```

### Die Verschiebe-Tabelle (Schlüssel k = 3)

Für einen Schlüssel `k = 3` ergibt sich folgende Ersetzungstabelle. Jede Spalte listet den Klartext-Buchstaben oben und den zugehörigen Geheimtext-Buchstaben unten:

| Klartext  | A | B | C | D | E | F | G | H | I | J | K | L | M |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Geheimtext | D | E | F | G | H | I | J | K | L | M | N | O | P |

| Klartext  | N | O | P | Q | R | S | T | U | V | W | X | Y | Z |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| Geheimtext | Q | R | S | T | U | V | W | X | Y | Z | A | B | C |

Beachte die drei letzten Spalten: Hier passiert das Umlaufen am Alphabet-Ende.

### Die Formel in einer Zeile

Mathematisch ausgedrückt läuft Caesar so: Jedem Buchstaben wird zunächst seine **Position im Alphabet** `pos` zugewiesen (`A`=0, `B`=1, …, `Z`=25). Dann gilt

$$ \text{neue\_pos} = (\text{pos} + k) \bmod 26 $$

Der Modulo-Operator `% 26` sorgt dafür, dass Werte ≥26 korrekt auf 0–25 zurückgefaltet werden. Anschließend wird `neue_pos` wieder in einen Buchstaben zurückübersetzt.

### Beispiel: `"HALLO"` + Schlüssel 3

Wir verschlüsseln das Wort `HALLO` Buchstabe für Buchstabe:

| Klartext | Position `pos` | `pos + 3` | `% 26` | Neue Position | Geheimtext |
|---|---|---|---|---|---|
| H | 7  | 10 | 10 | 10 | K |
| A | 0  |  3 |  3 |  3 | D |
| L | 11 | 14 | 14 | 14 | O |
| L | 11 | 14 | 14 | 14 | O |
| O | 14 | 17 | 17 | 17 | R |

Aus `HALLO` wird also `KDOOR`. Die Leerzeichen in längeren Sätzen lassen wir unverändert – sie sind keine Buchstaben A–Z.

### Caesar live in Python

Die Formel aus dem vorigen Abschnitt übersetzt sich direkt in eine kurze Python-Funktion. Wir arbeiten konsequent mit **Großbuchstaben** und ignorieren alle anderen Zeichen (Leerzeichen, Ziffern, Satzzeichen).

In [3]:
def caesar_zeichen(z, k):
    """Verschiebt einen Großbuchstaben z um k Schritte."""
    if "A" <= z <= "Z":
        pos = ord(z) - 65
        neu = (pos + k) % 26
        return chr(neu + 65)
    return z  # Leerzeichen, Satzzeichen, etc. bleiben unverändert

print(caesar_zeichen("H", 3))  # 'K'
print(caesar_zeichen("A", 3))  # 'D'
print(caesar_zeichen("Z", 3))  # 'C'  – der Umlauf
print(caesar_zeichen(" ", 3))  # ' '  – bleibt unverändert

K
D
C
 


Um einen ganzen Text zu verschlüsseln, wenden wir `caesar_zeichen` Buchstabe für Buchstabe an und sammeln das Ergebnis in einem neuen String.

In [4]:
def caesar_text(text, k):
    ergebnis = ""
    for z in text:
        ergebnis += caesar_zeichen(z, k)
    return ergebnis

print(caesar_text("HALLO WELT", 3))       # 'KDOOR ZHOW'
print(caesar_text("MASCHINE AN", 5))      # 'RFXHMNSJ FS'
print(caesar_text("ANGRIFF BEI MORGENGRAUEN", 7))

KDOOR ZHOW
RFXHMNSJ FS
HUNYPMM ILP TVYNLUNYHBLU


### Entschlüsseln = Verschlüsseln mit negativem Schlüssel

Wenn man um `k` nach rechts verschoben hat, muss man um `k` nach links zurückschieben. In der Formel ist das einfach `(pos - k) % 26`, und Python kann Modulo auch mit negativen Zahlen, sodass wir genauso `caesar_text(geheim, -k)` schreiben dürfen.

In [5]:
geheim = caesar_text("HALLO WELT", 3)
print("Geheim:", geheim)
print("Klar:  ", caesar_text(geheim, -3))   # 'HALLO WELT'

# Alternativ: (26 - 3) % 26 = 23 Schritte nach rechts = 3 Schritte nach links
print("Klar:  ", caesar_text(geheim, 23))

Geheim: KDOOR ZHOW
Klar:   HALLO WELT
Klar:   HALLO WELT


## 5. Warum Caesar unsicher ist: Brute-Force

Die Caesar-Chiffre hat genau **25 mögliche Schlüssel** (`k = 1` bis `k = 25`; `k = 0` oder `k = 26` wären identisch mit dem Klartext). Das bedeutet: Ein Angreifer probiert einfach alle 25 Schlüssel durch und schaut, welches Ergebnis lesbar aussieht. Dieses Vorgehen nennt man **Brute-Force-Angriff** – rohe Rechengewalt statt cleverer Mathematik.

Bei Caesar genügt dafür ein Mensch mit 25 Sekunden Zeit, oder ein Computer mit wenigen Mikrosekunden. Caesar ist damit **kryptografisch nutzlos** und dient heute nur noch als Lehrbeispiel – genau dafür verwenden wir es in dieser Vorlesung.

In [6]:
from IPython.display import Markdown, display
with open("diagramme/04_brute_force.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))

```mermaid
flowchart TD
    S(["Start: Geheimtext"]) --> K["k = 1"]
    K --> D["Mit Schluessel k entschluesseln"]
    D --> P["Ergebnis ausgeben"]
    P --> C{"k < 25?"}
    C -- "ja" --> I["k = k + 1"]
    I --> D
    C -- "nein" --> L["Mensch waehlt lesbaren Text"]
    L --> E(["Ende"])

```

In [7]:
# Brute-Force gegen einen Caesar-Geheimtext
geheim = "KDOOR ZHOW"
for k in range(1, 26):
    print(f"k = {k:2d}: {caesar_text(geheim, -k)}")

k =  1: JCNNQ YGNV
k =  2: IBMMP XFMU
k =  3: HALLO WELT
k =  4: GZKKN VDKS
k =  5: FYJJM UCJR
k =  6: EXIIL TBIQ
k =  7: DWHHK SAHP
k =  8: CVGGJ RZGO
k =  9: BUFFI QYFN
k = 10: ATEEH PXEM
k = 11: ZSDDG OWDL
k = 12: YRCCF NVCK
k = 13: XQBBE MUBJ
k = 14: WPAAD LTAI
k = 15: VOZZC KSZH
k = 16: UNYYB JRYG
k = 17: TMXXA IQXF
k = 18: SLWWZ HPWE
k = 19: RKVVY GOVD
k = 20: QJUUX FNUC
k = 21: PITTW EMTB
k = 22: OHSSV DLSA
k = 23: NGRRU CKRZ
k = 24: MFQQT BJQY
k = 25: LEPPS AIPX


In der Ausgabe oben siehst du für `k = 3` sofort den lesbaren Klartext `HALLO WELT`. Alle anderen 24 Zeilen sind Buchstabensalat. Das zeigt zwei Dinge:

1. **Brute-Force funktioniert bei kleinem Schlüsselraum trivial.**
2. Die Auswahl des richtigen Ergebnisses ist bei reinem ASCII-Text leicht, weil nur eine Zeile sinnvolle deutsche Wörter enthält.

Moderne Verfahren wie AES haben einen Schlüsselraum von $2^{128}$ oder $2^{256}$ – dagegen ist auch ein Rechenzentrum chancenlos.

> 💡 **Weiterführend:** Eine raffiniertere Variante ist die **Häufigkeitsanalyse**. Wer weiß, dass im Deutschen der Buchstabe `E` am häufigsten vorkommt, schaut im Geheimtext nach dem häufigsten Buchstaben und kennt sofort den Schlüssel. Das funktioniert bei allen **monoalphabetischen** Chiffren und war über Jahrhunderte das Standardwerkzeug von Kryptoanalytikern.

## 6. Symmetrisch vs. asymmetrisch – der große Unterschied

Alle modernen Verfahren lassen sich in zwei Familien einteilen, die sich darin unterscheiden, **wie viele Schlüssel im Spiel sind und wer sie kennt**. Der Unterschied hat massive Folgen für Schlüsselverwaltung, Performance und Einsatzgebiet.

### Symmetrische Verfahren

Ein **symmetrisches Verfahren** nutzt **einen einzigen Schlüssel** `K`, den sowohl Alice als auch Bob kennen. Alice verschlüsselt mit `K`, Bob entschlüsselt mit `K`. Die Caesar-Chiffre ist ein (unsicheres) symmetrisches Verfahren, denn der Verschiebewert ist für beide Seiten derselbe.

**Analogie Vorhängeschloss.** Stell dir einen verschließbaren Koffer vor, für den es nur einen Schlüssel gibt, von dem Alice und Bob jeweils eine Kopie besitzen. Alice packt die Nachricht hinein, schließt ab, Bob schließt auf. Das geht schnell, ist simpel – aber wie bekommt Bob überhaupt die Schlüsselkopie, wenn er in einem anderen Land sitzt?

**Praxis-Beispiel: AES.** Der *Advanced Encryption Standard* (AES) ist das meistgenutzte symmetrische Verfahren und arbeitet mit Schlüssellängen von 128, 192 oder 256 Bit. AES ist extrem schnell (moderne CPUs haben eigene AES-Instruktionen) und wird überall eingesetzt: Festplattenverschlüsselung, WLAN (WPA2), HTTPS-Datentransfer nach dem Handshake.

### Asymmetrische Verfahren

Ein **asymmetrisches Verfahren** nutzt **zwei zusammengehörende Schlüssel**: einen **öffentlichen** (*public key*) und einen **privaten** (*private key*). Was mit dem einen verschlüsselt wurde, lässt sich nur mit dem anderen wieder entschlüsseln. Bob veröffentlicht seinen Public Key für alle Welt, behält den Private Key ausschließlich bei sich.

**Analogie Briefkasten.** Bobs Briefkasten hat einen Schlitz (öffentlich, jeder kann einwerfen) und eine Klappe mit Schloss (privat, nur Bob hat den Schlüssel). Alice wirft ihre Nachricht ein – rausholen kann sie nur Bob. Der geniale Effekt: **Alice und Bob müssen vorher nie einen gemeinsamen Schlüssel ausgetauscht haben.**

**Praxis-Beispiel: RSA.** Der Algorithmus von *Rivest, Shamir und Adleman* (1977) beruht darauf, dass das Produkt zweier großer Primzahlen leicht zu berechnen, aber praktisch nicht zu faktorisieren ist. RSA-Schlüssel sind typischerweise 2048 oder 4096 Bit lang – und deutlich langsamer als AES.

In [8]:
from IPython.display import Markdown, display
with open("diagramme/03_symmetrisch_vs_asymmetrisch.mmd", encoding="utf-8") as f:
    display(Markdown(f"```mermaid\n{f.read()}\n```"))

```mermaid
flowchart LR
    subgraph SYM["Symmetrisch (z. B. AES)"]
        A1(["Alice"]) -- "Klartext + Schluessel K" --> V1["Verschluesseln"]
        V1 --> G1(["Geheimtext"])
        G1 --> E1["Entschluesseln<br/>mit gleichem K"]
        E1 --> B1(["Bob"])
    end
    subgraph ASYM["Asymmetrisch (z. B. RSA)"]
        A2(["Alice"]) -- "Klartext + Bobs<br/>oeffentlicher Schluessel" --> V2["Verschluesseln"]
        V2 --> G2(["Geheimtext"])
        G2 --> E2["Entschluesseln<br/>mit Bobs privatem Schluessel"]
        E2 --> B2(["Bob"])
    end

```

### Direkter Vergleich

| Eigenschaft | Symmetrisch (AES) | Asymmetrisch (RSA) |
|---|---|---|
| Anzahl Schlüssel | 1 gemeinsamer | 2 pro Person (public/private) |
| Geschwindigkeit | sehr schnell | deutlich langsamer (~1000×) |
| Schlüssellänge | 128–256 Bit | 2048–4096 Bit |
| Schlüsselaustausch | schwierig (Henne-Ei) | einfach (Public Key ist öffentlich) |
| Einsatzgebiet | große Datenmengen | kleine Daten, Schlüsselaustausch, Signaturen |

Aus der Tabelle liest man das **Henne-Ei-Problem** heraus: Symmetrisch ist schnell, aber der Austausch des Schlüssels ist heikel. Asymmetrisch löst den Austausch, ist aber zu langsam für GB-große Dateien. Die Lösung kombiniert beide Welten.

### Das Hybrid-Verfahren (z. B. bei HTTPS)

In der Praxis nutzt man ein **hybrides Verfahren**: Zu Beginn einer Verbindung wird per **asymmetrischer** Kryptografie ein zufälliger **Sitzungsschlüssel** ausgehandelt, mit dem dann per **symmetrischer** Kryptografie die eigentlichen Daten schnell verschlüsselt werden.

Bei **HTTPS / TLS** läuft das grob so ab:

1. Der Browser holt das TLS-Zertifikat des Servers – darin steht der öffentliche RSA-Schlüssel des Servers.
2. Der Browser würfelt einen AES-Sitzungsschlüssel und verschlüsselt ihn mit dem öffentlichen Server-Schlüssel.
3. Der Server entschlüsselt den AES-Schlüssel mit seinem privaten Schlüssel.
4. Ab jetzt läuft die gesamte Kommunikation per AES – schnell und trotzdem sicher, weil den AES-Schlüssel nur diese beiden Parteien kennen.

Moderne TLS-Versionen (1.3) ersetzen RSA durch **Diffie-Hellman mit elliptischen Kurven (ECDHE)**, das Grundprinzip bleibt aber dasselbe: asymmetrischer Austausch, dann symmetrische Massendaten.

## 7. Block- versus Stream-Chiffre (Kurzüberblick)

Innerhalb der symmetrischen Welt unterscheidet man zwei Bauarten:

- Eine **Block-Chiffre** verarbeitet den Klartext in Blöcken fester Größe (AES z. B. 128 Bit = 16 Byte pro Block). Kleinere Reststücke werden mit *Padding* aufgefüllt. Beispiele: AES, DES (veraltet), 3DES (veraltet).
- Eine **Stream-Chiffre** verschlüsselt Bit für Bit bzw. Byte für Byte, indem sie aus dem Schlüssel einen Pseudozufallsstrom ableitet und diesen per XOR mit dem Klartext kombiniert. Beispiele: ChaCha20, RC4 (veraltet).

In V17 implementieren wir keine dieser Familien – die Caesar-Chiffre arbeitet zeichenweise und ist am ehesten eine uralte Stream-Chiffre mit einem konstanten „Schlüsselstrom“. Wichtig ist, dass du die Begriffe einordnen kannst, wenn sie in Dokumentationen auftauchen.

## 8. Maschinenbau-Beispiel: CAD-Datei sicher übertragen

Stell dir vor, ein Ingenieurbüro in Stuttgart schickt einer Fertigung in Shanghai eine CAD-Zeichnung eines neuen Turbinenschaufelprofils. Die Datei ist 80 MB groß und streng vertraulich. So läuft der Versand in der Praxis:

1. Das Büro holt sich den **öffentlichen RSA-Schlüssel** der Fertigung aus deren Unternehmensverzeichnis.
2. Es würfelt einen zufälligen **AES-256-Schlüssel** (32 Byte).
3. Die CAD-Datei wird per **AES-GCM** verschlüsselt – in Sekunden, weil AES hardwarebeschleunigt ist.
4. Der AES-Schlüssel wird per **RSA** mit dem öffentlichen Schlüssel der Fertigung verschlüsselt.
5. Beides (verschlüsselte Datei + verschlüsselter Schlüssel) wird übertragen.
6. Die Fertigung entschlüsselt den AES-Schlüssel per **privatem RSA-Schlüssel** und danach die Datei per AES.

Das ist exakt das Hybrid-Prinzip aus dem vorigen Abschnitt – angewandt auf einen realen Industrie-Workflow. Genau so arbeiten auch Tools wie **PGP**, **S/MIME**, **SFTP mit Public-Key-Auth** und die meisten B2B-Transferportale.

## 9. Hinweis zu den `testprogramme/`

Im Ordner [testprogramme/](testprogramme/) liegen Python-Skripte aus V16, die per **Socket** kommunizieren (`P1_echo_client_TEST.py` etc.). Diese Programme **führen wir in V17 nicht aus**, weil sie nichts mit Kryptografie zu tun haben. Sie bleiben erhalten, damit du bei Bedarf zu V16 zurückspringen kannst. Alle Krypto-Beispiele in V17 laufen offline und erzeugen keinen Netzwerkverkehr.

## 10. Selbst-Check – Verständnisfragen

Beantworte die Fragen für dich selbst, bevor du die Lösungen aufklappst.

<details><summary><b>Frage 1:</b> Was genau ist bei Caesar der Schlüssel?</summary>

Der Schlüssel ist die ganze Zahl `k` zwischen 1 und 25, um die jeder Buchstabe verschoben wird. Bei `k = 3` wird aus `A` ein `D`, bei `k = 10` wird aus `A` ein `K`.
</details>

<details><summary><b>Frage 2:</b> Was besagt das Kerckhoffs-Prinzip in einem Satz?</summary>

Ein Verfahren muss sicher bleiben, selbst wenn der Algorithmus öffentlich bekannt ist – nur der Schlüssel darf geheim sein.
</details>

<details><summary><b>Frage 3:</b> Warum brauchen wir `% 26` in der Caesar-Formel?</summary>

Weil das Alphabet nach 26 Buchstaben endet und wir bei Überlauf wieder bei `A` anfangen müssen. `% 26` faltet jede Zahl auf den Bereich 0–25 zurück.
</details>

<details><summary><b>Frage 4:</b> Warum ist Caesar heute unsicher?</summary>

Weil der Schlüsselraum nur 25 Werte umfasst. Ein Angreifer probiert alle durch (Brute-Force) oder nutzt Häufigkeitsanalyse. In beiden Fällen ist die Chiffre in Sekunden gebrochen.
</details>

<details><summary><b>Frage 5:</b> Wie viele Schlüssel hat ein symmetrisches, wie viele ein asymmetrisches Verfahren?</summary>

Symmetrisch: **einen** gemeinsamen Schlüssel. Asymmetrisch: **zwei zusammengehörige** Schlüssel pro Person (einen öffentlichen und einen privaten).
</details>

<details><summary><b>Frage 6:</b> Warum nutzt HTTPS ein Hybrid-Verfahren statt nur RSA?</summary>

RSA ist für große Datenmengen zu langsam. Man nutzt RSA daher nur, um am Anfang einen kurzen AES-Schlüssel sicher auszutauschen, und verschlüsselt den eigentlichen Datenstrom schnell mit AES.
</details>

<details><summary><b>Frage 7:</b> Welche Rolle spielt ein Public Key bei RSA?</summary>

Mit dem Public Key verschlüsselt man Nachrichten an den Besitzer des zugehörigen Private Keys. Er darf und soll öffentlich zugänglich sein – das ist gerade der Witz des Verfahrens.
</details>

<details><summary><b>Frage 8:</b> Worin unterscheiden sich Block- und Stream-Chiffre?</summary>

Eine Block-Chiffre arbeitet auf Blöcken fester Größe (z. B. AES mit 128 Bit), eine Stream-Chiffre verarbeitet den Klartext bitweise/byteweise mit einem Pseudozufallsstrom (z. B. ChaCha20).
</details>

## ✅ Zusammenfassung

- **Kryptografie** schützt Vertraulichkeit, Integrität und Authentizität – im Ingenieur-Alltag z. B. bei CAD-Transfer und Maschinenkommandos.
- **Klartext**, **Geheimtext**, **Schlüssel**, **Verschlüsseln**, **Entschlüsseln** sind die Grundbegriffe.
- Das **Kerckhoffs-Prinzip** trennt öffentlichen Algorithmus von geheimem Schlüssel.
- Die **Caesar-Chiffre** folgt der Formel `(pos + k) % 26`, ist aber per Brute-Force in Sekunden gebrochen.
- **Symmetrisch** (ein Schlüssel, schnell, AES) und **asymmetrisch** (zwei Schlüssel, langsam, RSA) ergänzen sich im **Hybrid-Verfahren** (HTTPS).
- **Block-** und **Stream-Chiffren** sind die zwei Bauarten innerhalb der symmetrischen Welt.

## ➡️ Nächster Schritt
Weiter mit [02_praxis.ipynb](02_praxis.ipynb) – Caesar-Chiffre selbst programmieren.